# JSON Basics — 09: JSON to Parquet Pipeline

Production pipeline: JSON landing zone → validated Parquet analytics zone.

Topics: schema detection, multi-version handling, incremental processing,
partition by date, row count and checksum validation.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:57:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Create Multi-Day JSON Landing Zone



In [2]:

import json as pyjson, pathlib, random, datetime
random.seed(42)
LANDING = f"{DATA_DIR}/json_landing"
pathlib.Path(LANDING).mkdir(exist_ok=True)

CATS=["Electronics","Clothing","Books","Food","Sports"]
CTRS=["US","UK","DE","FR","JP"]

for day_offset in range(7):
    date = datetime.date(2024,3,1) + datetime.timedelta(days=day_offset)
    n = random.randint(8000,12000)
    rows=[{"order_id":day_offset*20000+i,"customer_id":random.randint(1,5000),
           "product":f"Product_{random.randint(1,100)}","category":random.choice(CATS),
           "country":random.choice(CTRS),"quantity":random.randint(1,10),
           "price":round(random.uniform(5,800),2),
           "status":random.choice(["pending","confirmed","shipped"]),
           "order_date":str(date)}
          for i in range(n)]
    path = f"{LANDING}/orders_{date}.json"
    with open(path,"w") as f: [f.write(pyjson.dumps(r)+"\n") for r in rows]
    print(f"  {date}: {n:,} rows → {path.split('/')[-1]}")


  2024-03-01: 10,619 rows → orders_2024-03-01.json
  2024-03-02: 10,220 rows → orders_2024-03-02.json
  2024-03-03: 10,645 rows → orders_2024-03-03.json
  2024-03-04: 10,127 rows → orders_2024-03-04.json
  2024-03-05: 9,012 rows → orders_2024-03-05.json
  2024-03-06: 11,000 rows → orders_2024-03-06.json
  2024-03-07: 11,615 rows → orders_2024-03-07.json


## Step 2 — Convert All Days to Parquet



In [3]:

from pyspark.sql.types import *
import glob as glib

schema = StructType([StructField(k,t) for k,t in [
    ("order_id",LongType()),("customer_id",LongType()),("product",StringType()),
    ("category",StringType()),("country",StringType()),("quantity",IntegerType()),
    ("price",DoubleType()),("status",StringType()),("order_date",DateType())]])

STORAGE = f"{DATA_DIR}/parquet_storage"
json_files = sorted(glib.glob(f"{LANDING}/orders_*.json"))

total_rows = 0
for json_file in json_files:
    date_str = json_file.split("orders_")[1].replace(".json","")
    df_day = spark.read.schema(schema).json(json_file) \
                  .withColumn("revenue", F.col("price")*F.col("quantity"))
    count = df_day.count()
    df_day.write.mode("overwrite") \
               .option("compression","zstd") \
               .partitionBy("category") \
               .parquet(f"{STORAGE}/date={date_str}")
    total_rows += count
    print(f"  {date_str}: {count:,} rows → Parquet (zstd, partitioned by category)")

print(f"\nTotal: {total_rows:,} rows ingested")


  2024-03-01: 10,619 rows → Parquet (zstd, partitioned by category)


  2024-03-02: 10,220 rows → Parquet (zstd, partitioned by category)


  2024-03-03: 10,645 rows → Parquet (zstd, partitioned by category)


  2024-03-04: 10,127 rows → Parquet (zstd, partitioned by category)


  2024-03-05: 9,012 rows → Parquet (zstd, partitioned by category)


  2024-03-06: 11,000 rows → Parquet (zstd, partitioned by category)


  2024-03-07: 11,615 rows → Parquet (zstd, partitioned by category)

Total: 73,238 rows ingested


## Step 3 — Incremental Processing



In [4]:

import pathlib

CHECKPOINT_FILE = f"{DATA_DIR}/json_checkpoint.txt"

def get_processed():
    try: return set(pathlib.Path(CHECKPOINT_FILE).read_text().strip().split("\n"))
    except FileNotFoundError: return set()

def mark_processed(filename):
    with open(CHECKPOINT_FILE,"a") as f: f.write(filename+"\n")

def run_incremental():
    all_files = sorted(glib.glob(f"{LANDING}/orders_*.json"))
    processed = get_processed()
    new_files  = [f for f in all_files if pathlib.Path(f).name not in processed]
    if not new_files:
        print("No new files."); return 0
    total=0
    for fp in new_files:
        date_str = fp.split("orders_")[1].replace(".json","")
        df_day   = spark.read.schema(schema).json(fp).withColumn("revenue",F.col("price")*F.col("quantity"))
        n        = df_day.count()
        df_day.write.mode("overwrite").option("compression","zstd") \
              .partitionBy("category").parquet(f"{STORAGE}/date={date_str}")
        mark_processed(pathlib.Path(fp).name)
        total+=n
        print(f"  Processed: {pathlib.Path(fp).name} ({n:,} rows)")
    return total

print("=== Run 1: initial ==="); run_incremental()

# Simulate new file
new_date = datetime.date(2024,3,8)
rows=[{"order_id":200000+i,"customer_id":random.randint(1,5000),"product":f"P_{i}",
       "category":random.choice(CATS),"country":random.choice(CTRS),
       "quantity":random.randint(1,5),"price":round(random.uniform(5,500),2),
       "status":"pending","order_date":str(new_date)} for i in range(3000)]
with open(f"{LANDING}/orders_{new_date}.json","w") as f: [f.write(pyjson.dumps(r)+"\n") for r in rows]

print("\n=== Run 2: new file ==="); run_incremental()
print("\n=== Run 3: nothing new ==="); run_incremental()


=== Run 1: initial ===


  Processed: orders_2024-03-01.json (10,619 rows)


  Processed: orders_2024-03-02.json (10,220 rows)


  Processed: orders_2024-03-03.json (10,645 rows)


  Processed: orders_2024-03-04.json (10,127 rows)


  Processed: orders_2024-03-05.json (9,012 rows)


  Processed: orders_2024-03-06.json (11,000 rows)


  Processed: orders_2024-03-07.json (11,615 rows)

=== Run 2: new file ===


  Processed: orders_2024-03-08.json (3,000 rows)

=== Run 3: nothing new ===
No new files.


0

## Step 4 — Validation and Analytics



In [5]:

total_json = sum(spark.read.schema(schema).json(f).count()
                 for f in glib.glob(f"{LANDING}/orders_*.json"))
total_pq   = spark.read.parquet(STORAGE).count()
print(f"Validation:")
print(f"  JSON total  : {total_json:,}")
print(f"  Parquet total: {total_pq:,}")
print(f"  Match       : {'✅' if total_json==total_pq else '❌'}")
print()
spark.read.parquet(STORAGE).groupBy("category") \
     .agg(F.sum("revenue").alias("revenue"),F.count("*").alias("orders")) \
     .orderBy(F.desc("revenue")).show()


Validation:
  JSON total  : 76,238
  Parquet total: 76,238
  Match       : ✅



[Stage 89:>                                                         (0 + 4) / 4]

+-----------+--------------------+------+
|   category|             revenue|orders|
+-----------+--------------------+------+
|     Sports|3.3490383049999975E7| 15299|
|      Books|3.2989583539999977E7| 15360|
|   Clothing| 3.293109199000001E7| 15266|
|       Food|       3.283829222E7| 15217|
|Electronics|3.2386578079999983E7| 15096|
+-----------+--------------------+------+



## Summary



In [6]:

print("""
JSON → Parquet pipeline:
  1. Identify new JSON files (checkpoint log)
  2. Read with explicit schema
  3. Add derived columns (revenue = price * quantity)
  4. Write partitioned Parquet (compression=zstd)
  5. Validate row counts
  6. Mark file as processed in checkpoint

Checkpoint pattern:
  - Simple: text file with one processed filename per line
  - Production: Delta/Iceberg table with metadata
  - Streaming: use Spark Structured Streaming checkpoints
""")



JSON → Parquet pipeline:
  1. Identify new JSON files (checkpoint log)
  2. Read with explicit schema
  3. Add derived columns (revenue = price * quantity)
  4. Write partitioned Parquet (compression=zstd)
  5. Validate row counts
  6. Mark file as processed in checkpoint

Checkpoint pattern:
  - Simple: text file with one processed filename per line
  - Production: Delta/Iceberg table with metadata
  - Streaming: use Spark Structured Streaming checkpoints

